## 3CX CALLING REPORT

In [780]:
import pandas as pd
import numpy as np
import gspread
from gspread_pandas import Spread, conf
from datetime import datetime
import pytz

In [781]:
# Use read_csv for .csv files
df = pd.read_csv(r"C:\Users\USER\Downloads\Emarath_global\3CX_Calling_Report_Dataset\call_reports.csv")

In [782]:
df['username'] = df['From'].str.split(' \\(').str[0]
df_filtered = df[df['username'].str.lower().str.endswith('global', na=False)]
df_filtered.head(2)

,Call Time,Call ID,From,To,Direction,Status,Ringing,Talking,Cost,Call Activity Details,Sentiment,Summary,Transcription,username
0,2026-04-07T17:48:25,00000000-01dc-c688-a20c-c3d200003061,Shihad Emarathglobal (133),00966505882693,Outbound,Unanswered,00:00:31,00:00:00,0.0,Dialed: (00966505882693) → Via rule: Outbound...,NaN,NaN,NaN,Shihad Emarathglobal
2,2026-04-07T17:48:14,00000000-01dc-c688-9bf7-7ece0000305f,Shihad Emarathglobal (133),00966505882693,Outbound,Unanswered,00:00:00,00:00:00,0.0,Dialed: (00966505882693) → Via rule: Outbound...,NaN,NaN,NaN,Shihad Emarathglobal


In [783]:
df_filtered['Call Time'] = pd.to_datetime(df_filtered['Call Time'])
df_filtered['Date'] = df_filtered['Call Time'].dt.date

In [784]:
df_filtered['dialed count'] = df_filtered.groupby(['Call ID', 'From'])['To'].transform('nunique')
df_new = df_filtered[[ 'From','To', 'dialed count', 'Status', 'Direction','Call Activity Details']].copy()
df_summary = df_new

In [785]:
# Wrong number (Indian Number)
df_filtered['To'] = df_filtered['To'].astype(str).str.replace('+', '').str.strip()
is_indian = df_filtered['To'].str.startswith('91') | df_filtered['To'].str.startswith('0091')
indian_numbers_df = df_filtered[is_indian]
indian_count = len(indian_numbers_df)

print(f"Total Indian numbers found: {indian_count}")
indian_numbers_df[['Call Time', 'To', 'username']].head(2)

Total Indian numbers found: 0


,Call Time,To,username


In [786]:
status_counts = df_filtered.groupby('From')['Status'].value_counts().unstack(fill_value=0)
status_counts = status_counts[['Answered', 'Unanswered']]  
status_counts['Dailed Count'] = status_counts['Answered'] + status_counts['Unanswered']
# status_counts.head(2)

In [787]:
df_filtered['Talking_seconds'] = pd.to_timedelta(df_filtered['Talking']).dt.total_seconds()
total_duration = df_filtered.groupby('From')['Talking_seconds'].sum().reset_index()
total_duration['Total Duration (minutes)'] = total_duration['Talking_seconds'] / 60
total_duration = total_duration.rename(columns={'From': 'User'})[['User', 'Total Duration (minutes)']]
total_duration['Total Duration (minutes)']=total_duration['Total Duration (minutes)'].round(2)

In [788]:
total_seconds = df_filtered.groupby('From')['Talking_seconds'].sum()
answered_calls = status_counts['Answered']
average_duration_minutes = (total_seconds / 60).div(answered_calls).replace([float('inf'), -float('inf')], 0).fillna(0).round(2)
avg_df = average_duration_minutes.reset_index()
avg_df.columns = ['User', 'Avg Minutes per Answered Call']

In [789]:
status_counts_reset = status_counts.reset_index()
merged_df = status_counts_reset.merge(total_duration, left_on='From', right_on='User', how='left').drop('User', axis=1)
merged_df = merged_df.merge(avg_df, left_on='From', right_on='User', how='left').drop('User', axis=1)

In [790]:
SUPER_SALES_List = ['Chaithanya Emarathglobal (127)','Habiya Emarathglobal (122)', 'Amina Emarathglobal (120)', 'Rahiyad Emarathglobal (136)', 'Nihad Emarathglobal (129)', 'Safwan K P Emarathglobal (137)']
BDM1_List = ['SHAMNA Emarathglobal (132)','Adwaitha Emarathglobal (134)', 'Rahib Emarathglobal (125)', 'Ayaana Emarathglobal (131)', 'Ranjith Emarathglobal (119)', 'Afnan Emarathglobal (130)']
BDM2_List = []
BDM3_List = ['Arshad Emarathglobal (121)','Zakiya Emarathglobal (138)','Shihad Emarathglobal (133)', 'Najiya Emarathglobal (124)','Akash Emarathglobal (123)','Neha Emarathglobal (128)']
SUPER_SALES_df = merged_df[merged_df['From'].isin(SUPER_SALES_List)]
BDM1_df = merged_df[merged_df['From'].isin(BDM1_List)]
BDM2_df = merged_df[merged_df['From'].isin(BDM2_List)]
BDM3_df = merged_df[merged_df['From'].isin(BDM3_List)]

In [791]:
names_to_filter = [ "Vaishakh Emarathglobal (126)", "Ihsan Emarathglobal (139)"]
logi_call_report_df = merged_df[merged_df['From'].isin(names_to_filter)]

In [792]:
blacklist = [
    "Vaishakh Emarathglobal (126)",
    'AJITH T R Emarathglobal (140)',
    "Ihsan Emarathglobal (139)"
]

merged_df = merged_df[~merged_df['From'].isin(blacklist)]
merged_df = merged_df[~merged_df['From'].str.lower().isin([x.lower() for x in blacklist])]

col_to_move = merged_df.pop('Dailed Count')
merged_df.insert(1, 'Dailed Count', col_to_move)

In [793]:
col_to_move = logi_call_report_df.pop('Dailed Count')
logi_call_report_df.insert(1, 'Dailed Count', col_to_move)
logi_call_report_df.head(3)

,From,Dailed Count,Answered,Unanswered,Total Duration (minutes),Avg Minutes per Answered Call
4,Ihsan Emarathglobal (139),186,39,147,29.53,0.76
13,Vaishakh Emarathglobal (126),88,37,51,37.03,1.00


In [794]:
col_to_move = SUPER_SALES_df.pop('Dailed Count')
SUPER_SALES_df.insert(1, 'Dailed Count', col_to_move)
col_to_move = BDM1_df.pop('Dailed Count')
BDM1_df.insert(1, 'Dailed Count', col_to_move)
col_to_move = BDM2_df.pop('Dailed Count')
BDM2_df.insert(1, 'Dailed Count', col_to_move)
col_to_move = BDM3_df.pop('Dailed Count')
BDM3_df.insert(1, 'Dailed Count', col_to_move)
# BDM1_df.head(2)

In [795]:
merged_df.rename(columns={'From': 'BDE'}, inplace=True)
merged_df.head(2)

,BDE,Dailed Count,Answered,Unanswered,Total Duration (minutes),Avg Minutes per Answered Call
0,Adwaitha Emarathglobal (134),124,26,98,62.28,2.40
1,Akash Emarathglobal (123),154,36,118,37.33,1.04


In [796]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
c = conf.get_config(file_name=config_path)


# report_sheet_url = "https://docs.google.com/spreadsheets/d/1OU786oBbozzYkvr2O9WUG3IECJfLgxVwDOt_lZRsAKo/edit?gid=549469363#gid=549469363"
report_sheet_url = "https://docs.google.com/spreadsheets/d/1PqMNtFU0bas_BGYFhIYWojO2petW-TOGxeRB2OWnF08/edit?gid=113745152#gid=113745152"

PENDING_SHEET_URL = "https://docs.google.com/spreadsheets/d/1OU786oBbozzYkvr2O9WUG3IECJfLgxVwDOt_lZRsAKo/edit?gid=549469363#gid=549469363" 
PENDING_SPREAD = Spread(PENDING_SHEET_URL, config=c)
spread = Spread(report_sheet_url, config=c)

Data saved to user_call_summary_07-04-26.xlsx
The report will be saved to tab: 3cx_report_07-04-26
Data successfully pushed to tab: 3cx_report_07-04-26


In [ ]:

tab_name = "logistic_call_report"
logi_report_date = datetime.now().strftime("%Y-%m-%d")
LOGISTIC_DATA = logi_call_report_df.copy()

if 'Report Date' not in logi_call_report_df.columns:
    logi_call_report_df.insert(0, 'Report Date', logi_report_date)

try:
    logi_spread = Spread(report_sheet_url, config=c)
    logi_spread.open_sheet(tab_name, create=False)

    existing_df = logi_spread.sheet_to_df()
    is_empty = existing_df.empty
    
    current_data_rows = len(existing_df)
    start_row = 1 if is_empty else current_data_rows + 2

    logi_spread.df_to_sheet(
        logi_call_report_df, 
        index=False, 
        replace=False, 
        headers=is_empty,
        start=f'A{start_row}' 
    )

    print(f"Data successfully appended to '{tab_name}' starting at row {start_row}.")

except Exception as e:
    print(f"Detailed Error: {e}")

Data successfully appended to 'logistic_call_report' starting at row 7.


In [ ]:
report_date = datetime.now()
# yesterday = datetime.now() - timedelta(days=1)
date_str = report_date.strftime("%d-%m-%y")

# Save to Excel
file_path = f"./Output_Data/user_call_summary_{date_str}.xlsx"
merged_df.to_excel(file_path, index=False)
print(f"Data saved to user_call_summary_{date_str}.xlsx")

dynamic_tab_name = f"3cx_report_{date_str}"
print(f"The report will be saved to tab: {dynamic_tab_name}")

spread.open_sheet(dynamic_tab_name, create=True)
spread.df_to_sheet(merged_df, index=False, replace=True, start="A3")
PENDING_SPREAD.open_sheet("AGENT CALL REPORT", create=True)
PENDING_SPREAD.df_to_sheet(merged_df, index=False, replace=True, start="A2")
PENDING_SPREAD.open_sheet("LOGISTICS CALL REPORT", create=True)
PENDING_SPREAD.df_to_sheet(LOGISTIC_DATA, index=False, replace=True, start="A2")
print(f"Data successfully pushed to tab: {dynamic_tab_name}")

In [799]:
BDM_sheet_url = "https://docs.google.com/spreadsheets/d/19B1Vclt9U0AOu-yk99E960g5tYRQ9JRX-M2GlAPvE6U/edit#gid=0"

tabs_to_process = {
    "SUPER_SALES" : SUPER_SALES_df,
    "BDM1" : BDM1_df,
    "BDM2" : BDM2_df,
    "BDM3" : BDM3_df
}


ist = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)
call_report_date = now_ist.strftime("%Y-%m-%d")
call_report_time = now_ist.strftime("%H:%M")


try:
    bdm_spread = Spread(BDM_sheet_url, config=c)
    
    for tab_name, df in tabs_to_process.items():
        cols_to_add = {'Report Date': call_report_date, 'Report Time': call_report_time}
        
        for idx, (col_name, value) in enumerate(cols_to_add.items()):
            if col_name not in df.columns:
                df.insert(idx, col_name, value)
            else:
                df[col_name] = value

        bdm_spread.open_sheet(tab_name, create=True)
        # PENDING_SPREAD.open_sheet(tab_name, create=True)
        bdm_spread.sheet.batch_clear(['A2:Z10000']) 
        # PENDING_SPREAD.sheet.batch_clear(['A2:Z10000'])
        upload_df = df.astype(str)
        bdm_spread.df_to_sheet(upload_df, index=False, headers=False, start="A2")
        # PENDING_SPREAD.df_to_sheet(upload_df, index=False, headers=False, start="A2")
        
        print(f"✅ {tab_name} refreshed for {call_report_date}.")

    print(f"Success: All {len(tabs_to_process)} tabs updated.")

except Exception as e:
    print(f"❌ Detailed Error: {e}")

✅ SUPER_SALES refreshed for 2026-04-07.
✅ BDM1 refreshed for 2026-04-07.
❌ Detailed Error: list index out of range
